# Pydantic AI weather-sub agent diagnostics

This notebook intentionally ignores the project prompts. 

the agent should take a sequence of locations and search call for weather provider
than it should return it with small summery


In [2]:
import json
import asyncio
import os
from pathlib import Path
from pprint import pprint

from pydantic import BaseModel, Field
from pydantic_ai import Agent

from capabilities.weather import WeatherCapability
from capabilities.weather import default_weather_fn
from core.models import StargazingSpot, WeatherReport



async def run_with_timeout(label: str, awaitable, timeout: float = 45):
    print(f"START: {label}")
    result = await asyncio.wait_for(awaitable, timeout=timeout)
    print(f"DONE: {label}")
    return result


def load_dotenv(path: str = ".env") -> None:
    env_path = Path(path)
    if not env_path.exists():
        return
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ[key] = value


load_dotenv()

MODEL = os.environ["LAZY_STELLAR_MODEL"]

print(f"MODEL: {MODEL}")
for key in ["OPENROUTER_API_KEY", "OPENAI_API_KEY", "TAVILY_API_KEY"]:
    print(f"{key}: {'set' if os.getenv(key) else 'missing'}")


MODEL: openrouter:google/gemini-3.1-flash-lite
OPENROUTER_API_KEY: set
OPENAI_API_KEY: missing
TAVILY_API_KEY: set


In [ ]:
class WeatherReport(BaseModel):
    summary: str = Field(description="Geneeric summery for the rigion") 
    spots: list[WeatherReport] = Field(defaut_factory=True)
    note: str = None #descri}ption="Geneeric summery for the rigion"

In [ ]:
test_fake_reguest = {'spots': [{'accessibility': 'Accessible via Transilien line R from Gare de '
                             'Lyon (approx. 40-50 min). A short walk or '
                             'shuttle is required to reach the forest edges '
                             'from the station.',
            'additional_info': None,
            'bortle_class': 4,
            'description': 'A vast, historic forest region southeast of Paris. '
                           'Its significant distance from the city center '
                           'provides a much darker sky than the urban core, '
                           'making it a popular choice for astronomy '
                           'enthusiasts seeking better visibility.',
            'latitude': 48.4286,
            'longitude': 2.7001,
            'name': 'Fontainebleau Forest',
            'safety_assessment': 'Generally safe, but requires basic '
                                 'wilderness awareness (bring a flashlight, '
                                 'stay on trails, and dress for night '
                                 'temperatures). Avoid heavily wooded areas '
                                 'alone at night.',
            'seasonal_nature_risks': None,
            'source': 'GeneralAN.',
            'time_of_discovery': None},
           {'accessibility': 'Easily accessible via Metro Line 1 (Château de '
                             'Vincennes) or RER A (Fontenay-sous-Bois or '
                             'Vincennes stations).',
            'additional_info': None,
            'bortle_class': 7,
            'description': 'The largest public park in Paris, located on the '
                           'eastern edge of the city. While still affected by '
                           'city light pollution, its vast open spaces offer a '
                           'better vantage point than the dense city streets '
                           'for observing major constellations and planets.',
            'latitude': 48.8315,
            'longitude': 2.4414,
            'name': 'Bois de Vincennes',
            'safety_assessment': 'Public park; generally safe, but best to '
                                 'stay near well-lit pathways or within groups '
                                 "at night. The park's openness provides good "
                                 'visibility of surroundings.',
            'seasonal_nature_risks': None,
            'source': 'Urban planning and local Paris activity guides.',
            'time_of_discovery': None},
           {'accessibility': 'Accessible via Transilien Line N from Gare '
                             'Montparnasse (approx. 35-50 min). The station is '
                             'close to the outskirts of the forest.',
            'additional_info': None,
            'bortle_class': 4,
            'description': 'A large forest southwest of Paris known for its '
                           'significantly reduced light pollution compared to '
                           'the city. It provides a quiet, natural environment '
                           'suitable for stargazing once you move away from '
                           'the town center.',
            'latitude': 48.6432,
            'longitude': 1.8341,
            'name': 'Rambouillet Forest',
            'safety_assessment': 'Generally safe. As with all forest '
                                 'locations, it is recommended to remain on '
                                 'marked trails and maintain situational '
                                 'awareness, especially when accessing the '
                                 'site after dark.',
            'seasonal_nature_risks': None,}]
}

# 0 - just testing plain call

In [ ]:
weather_resolver_agent = Agent(
    model=MODEL,
    defer_model_check=True,
    capabilities=[WeatherCapability()]
)

result = await weather_resolver_agent.run("Hi how are you?")

pprint(result)

In [ ]:
type(result)

## 1. Sub - agent assembling and test
If this hangs or fails, the problem is model/provider configuration, not search or structured output.


In [13]:
weather_resolver_agent = Agent(
    model=MODEL,
    defer_model_check=True,
    instructions="Using tool with 7timer API, get the weather report today's night for given locations, return the structured output, as list[WeatherReport], for each place. if there is som trubles with fetching weather, add note ",
    capabilities=[WeatherCapability()]
)

plain_result = await weather_resolver_agent.run("how what about the weather in Paris?, if there is some troubles with fetching weather, try using latitude: 48.4286," \
"longitude: 2.7001,   tell me how many times you will try to fetch the API - and what codes it will return")


print("OUTPUT:")
print(plain_result.output)
print("\nUSAGE:")
print(plain_result.usage)


OUTPUT:
The weather information for Paris was successfully fetched on the first attempt without any errors. Since the API request was successful immediately, there were no secondary attempts required, and therefore no specific error codes were encountered.

Here is the weather report for Paris for tonight (based on the provided forecast):

### **Weather Report: Paris**
*   **Tonight (May 24th/25th):**
    *   **Cloud Cover:** Mostly clear with low cloud cover (ranging from 2/8 to 4/8).
    *   **Precipitation:** None expected.
    *   **Temperature:** Cooling down from 21°C towards 17°C by early morning.
    *   **Wind:** Light winds from the Northeast at a speed of 2 units.
    *   **Visibility/Conditions:** Favorable atmospheric conditions for viewing, with moderate transparency.

*(Note: Data is based on the 7timer astro-forecast integration.)*

USAGE:
RunUsage(input_tokens=2229, output_tokens=247, details={'is_byok': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'image_tokens': 0}, 

SyntaxError: unterminated string literal (detected at line 8) (1054580087.py, line 8)

: 

## 2. Real search tool, plain text output

If step 1 works but this fails, inspect whether the model called `search_spots`, whether Tavily/DuckDuckGo failed, or whether the tool result came back as an error string.


In [ ]:
search_text_agent = Agent(
    model=MODEL,
    defer_model_check=True,
    capabilities=[SearchCapability()],
    instructions=(
        "You are a stargazing assistant. For location-specific recommendations, "
        "call search_spots exactly once before answering. Then summarize the returned spots. "
        "If the tool returns an error string, report that exact failure briefly."
    ),
)

search_text_result = await run_with_timeout(
    "search tool + plain text output",
    search_text_agent.run(
        "Find 3 stargazing spots near Paris, reachable without a private car."
    ),
    timeout=90,
)

print("OUTPUT:")
print(search_text_result.output)
print("\nUSAGE:")
print(search_text_result.usage)
print("\nMESSAGES:")
for message in search_text_result.all_messages():
    print(type(message).__name__, message)


## 2a. Real search tool, JSON text output

This tests whether the model can use the search tool and emit JSON text when Pydantic AI does not force the final `output_type` tool.


In [2]:
from pydantic_ai.capabilities import WebSearch
from pydantic_ai.capabilities import Thinking


json_text_agent = Agent(
    model=MODEL,
    defer_model_check=True,
    # capabilities=[SearchCapability()],
    capabilities=[WebSearch(), Thinking("high")],
    instructions=(
        "You are a stargazing assistant. You must call search_spots exactly once before final output. "
        "Then return ONLY valid JSON text with keys: spots_found, summary, spots. "
        "spots must be an array of objects with name, latitude, longitude, source, description, accessibility, safety_assessment, and bortle_class. "
        "If search_spots returns an error string, return {\"spots_found\": 0, \"summary\": <error>, \"spots\": []}. "
        "Do not wrap the JSON in markdown. Do not invent spots that were not returned by the tool."
    ),
)

json_text_result = await run_with_timeout(
    "search tool + JSON text output",
    json_text_agent.run(
        "Find 3 stargazing spots near Paris, reachable without a private car."
    ),
    timeout=90,
)

print("RAW OUTPUT:")
print(json_text_result.output)
print("\nPARSED JSON:")
pprint(json.loads(json_text_result.output))
print("\nUSAGE:")
print(json_text_result.usage)
print("\nMESSAGES:")
for message in json_text_result.all_messages():
    print(type(message).__name__, message)


/var/folders/lt/75xqxt7561731l87vw1h2_3w0000gn/T/ipykernel_5619/4026942807.py:9: PydanticAIDeprecationWarning: WebSearch will stop auto-selecting DuckDuckGo based on package availability in v2. To keep this fallback, pass `local='duckduckgo'` (or `local=True`). To disable the fallback, pass `local=False`.
  capabilities=[WebSearch(), Thinking("high")],
Impersonate 'chrome_101' does not exist, using 'random'


START: search tool + JSON text output
DONE: search tool + JSON text output
RAW OUTPUT:
{
  "spots_found": 3,
  "summary": "While Paris itself suffers from significant light pollution, several natural areas within the Île-de-France region, accessible via the Transilien train network, provide dark enough skies for casual stargazing. The Fontainebleau and Rambouillet forests are recognized for their vastness and relative isolation, while the high-altitude Limours plateau offers an elevated vantage point.",
  "spots": [
    {
      "name": "Forêt de Rambouillet",
      "latitude": 48.64,
      "longitude": 1.83,
      "source": "https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEP2lI6JtT7xRGQEjnf9GVWfCIaXF4ueFUTcn8lg7cB8jVwTUFvQqeJJAh3LQ_eBMxYI-3mHXi7dxB7YqUjnBcF4kUiRKngr7MiQ3OV6izTF9DAOEOCI00Lv3axO8caLNpA5z-_nC7VOJ0Uy_m4zQ==",
      "description": "A vast, expansive forest located south-west of Paris. Its size and distance from the city center help reduce light pollutio

## 3. Real search tool, structured Pydantic output

If steps 1 and 2 work but this fails, the problem is structured output/tool interaction. This cell forces a `SpotSearchReport` final result.


In [ ]:
structured_agent = Agent(
    model=MODEL,
    defer_model_check=True,
    output_type=SpotSearchReport,
    capabilities=[WebSearch(), Thinking(effort=True)],
    instructions=(
        "You are a stargazing assistant. You must call search_spots exactly once before final output. "
        "Use the tool result to fill SpotSearchReport. "
        "If search_spots returns an error string, return spots_found=0, spots=[], and put the error in summary. "
        "Do not invent spots that were not returned by the tool."
        "before returning rechack all fields for correctness and consistency, and if you find any issues, use the search tool again"
        "You can use search tool max 2 times for each spot"
        "Add the UTC offset for each spot based on location"
    ),
)

structured_result = await run_with_timeout(
    "search tool + structured Pydantic output",
    structured_agent.run(
        "Find 3 stargazing spots near Paris, reachable without a private car."
    ),
    timeout=20,
)

print("PYDANTIC OUTPUT:")
pprint(structured_result.output.model_dump())
print("\nJSON:")
print(structured_result.output.model_dump_json(indent=2))
print("\nUSAGE:")
print(structured_result.usage)
print("\nMESSAGES:")
for message in structured_result.all_messages():
    print(type(message).__name__, message)


In [ ]:
# checking Api function

In [10]:
result = default_weather_fn(longitude=2.7001, latitude=48.8566)

In [11]:
pprint(result)

WeatherReport(name='Astro weather at (48.8566, 2.7001)',
              latitude=48.8566,
              longitude=2.7001,
              cloud_cover={'00:00': 1,
                           '03:00': 1,
                           '06:00': 1,
                           '09:00': 1,
                           '21:00': 1},
              transparency={'00:00': 3,
                            '03:00': 4,
                            '06:00': 3,
                            '09:00': 3,
                            '21:00': 3},
              seeing={'00:00': 5,
                      '03:00': 5,
                      '06:00': 3,
                      '09:00': 3,
                      '21:00': 5},
              wind_speed={'00:00': 2,
                          '03:00': 2,
                          '06:00': 2,
                          '09:00': 2,
                          '21:00': 2},
              wind_direction={'00:00': 'NE',
                              '03:00': 'N',
                              '

In [ ]:
print(2+2)


In [1]:
# simple Api call

import httpx


url = "http://www.7timer.info/bin/api.pl?lon=113.17&lat=23.09&product=astro&output=json"

response = httpx.get(url)

In [2]:
print(response.text)

{
	"product" : "astro" ,
	"init" : "2026052512" ,
	"dataseries" : [
	{
		"timepoint" : 3,
		"cloudcover" : 5,
		"seeing" : 6,
		"transparency" : 5,
		"lifted_index" : -1,
		"rh2m" : 11,
		"wind10m" : {
			"direction" : "S",
			"speed" : 3
		},
		"temp2m" : 30,
		"prec_type" : "none"
	},
	{
		"timepoint" : 6,
		"cloudcover" : 3,
		"seeing" : 6,
		"transparency" : 6,
		"lifted_index" : -1,
		"rh2m" : 12,
		"wind10m" : {
			"direction" : "S",
			"speed" : 3
		},
		"temp2m" : 29,
		"prec_type" : "none"
	},
	{
		"timepoint" : 9,
		"cloudcover" : 2,
		"seeing" : 6,
		"transparency" : 6,
		"lifted_index" : -1,
		"rh2m" : 12,
		"wind10m" : {
			"direction" : "S",
			"speed" : 3
		},
		"temp2m" : 28,
		"prec_type" : "none"
	},
	{
		"timepoint" : 12,
		"cloudcover" : 3,
		"seeing" : 6,
		"transparency" : 5,
		"lifted_index" : -1,
		"rh2m" : 10,
		"wind10m" : {
			"direction" : "S",
			"speed" : 3
		},
		"temp2m" : 30,
		"prec_type" : "none"
	},
	{
		"timepoint" : 15,
		"cloudcover" : 9,
		"seein

In [9]:
def get_sum(a,b):
    #good luck!
    if a == b:
        return a
    elif a < b:
        small=a
        big=b
    elif b < a:
        small=b
        big=a
        
    
    answer = 0
   
    while small != big:
        answer += small
        small += 1
        
    return answer
        

In [12]:
print(get_sum(0, 3))

3


In [6]:
a=1
b=5

if a < b:
    small=a
    big=b
elif b < a:
    small=b
    big=a

print(small, big)

1 5


In [24]:
from capabilities.weather import WeatherCapability
from core.models import LocationQuery

# 1. Создаем список локаций (Париж и Хьюстон) c координатами и таймзонами
locations = [
    LocationQuery(
        name="Paris",
        latitude=48.8566,
        longitude=2.3522,
        timezone_offset=2.0  # UTC+2 (летнее время в Париже)
    ),
    LocationQuery(
        name="Houston",
        latitude=29.7604,
        longitude=-95.3698,
        timezone_offset=-5.0 # UTC-5 (летнее время CDT в Хьюстоне)
    )
]

# 2. Инициализируем WeatherCapability и достаем функцию инструмента
capability = WeatherCapability()
toolset = capability.get_toolset()
get_astro_weather_tool = toolset.tools["get_astro_weather"].function

# 3. Вызываем инструмент (первым параметром передаем None вместо ctx)
print("Fetching weather in parallel...")
results = get_astro_weather_tool(None, locations=locations)

# 4. Красиво печатаем результаты
for loc_name, report in results.items():
    print(f"\n======================================")
    print(f"📍 Location: {loc_name}")
    print(f"======================================")
    
    if isinstance(report, str):
        # Если произошла ошибка при запросе к этой локации
        print(f"❌ {report}")
    else:
        # Успешный отчет WeatherReport
        print(f"Coordinates: Lat {report.latitude}, Lon {report.longitude}")
        print(f"Forecast points count: {len(report.forecasts)}")
        
        # Выводим первую и последнюю точки прогноза для демонстрации диапазона времени
        if report.forecasts:
            print("\n--- First night point (Upcoming):")
            f_start = report.forecasts[0]
            print(f"  Local Time: {f_start.time}")
            print(f"  Cloud Cover (1-9): {f_start.cloud_cover}")
            print(f"  Seeing (1-8): {f_start.seeing}")
            print(f"  Temperature: {f_start.temperature}°C")
            print(f"  Wind: {f_start.wind_speed} (dir: {f_start.wind_direction})")
            print(f"  Precipitation: {f_start.precipitation}")

            print("\n--- Next night points (Times):")
            times = [f.time for f in report.forecasts[1:]]
            print("  " + ", ".join(times))


Fetching weather in parallel...

📍 Location: Paris
Coordinates: Lat 48.8566, Lon 2.3522
Forecast points count: 8

--- First night point (Upcoming):
  Local Time: 05-26 23:00
  Cloud Cover (1-9): 1
  Seeing (1-8): 6
  Temperature: 20.0°C
  Wind: 2 (dir: NE)
  Precipitation: none

--- Next night points (Times):
  05-27 02:00, 05-27 05:00, 05-27 08:00, 05-27 23:00, 05-28 02:00, 05-28 05:00, 05-28 08:00

📍 Location: Houston
Coordinates: Lat 29.7604, Lon -95.3698
Forecast points count: 8

--- First night point (Upcoming):
  Local Time: 05-26 22:00
  Cloud Cover (1-9): 9
  Seeing (1-8): 7
  Temperature: 27.0°C
  Wind: 3 (dir: SE)
  Precipitation: none

--- Next night points (Times):
  05-27 01:00, 05-27 04:00, 05-27 07:00, 05-27 22:00, 05-28 01:00, 05-28 04:00, 05-28 07:00


In [20]:
type(results)

dict

In [37]:
pprint(results, indent = 1, width= 4 )

{'Houston': WeatherReport(name='Houston', latitude=29.7604, longitude=-95.3698, forecasts=[HourlyForecast(time='05-26 22:00', cloud_cover=9, transparency=6, seeing=7, lifted_index=-4, wind_speed=3, wind_direction='SE', temperature=27.0, humidity=13, precipitation='none'), HourlyForecast(time='05-27 01:00', cloud_cover=9, transparency=7, seeing=7, lifted_index=-6, wind_speed=3, wind_direction='SE', temperature=25.0, humidity=14, precipitation='rain'), HourlyForecast(time='05-27 04:00', cloud_cover=9, transparency=6, seeing=7, lifted_index=-6, wind_speed=2, wind_direction='S', temperature=25.0, humidity=14, precipitation='rain'), HourlyForecast(time='05-27 07:00', cloud_cover=9, transparency=5, seeing=8, lifted_index=-4, wind_speed=2, wind_direction='S', temperature=24.0, humidity=14, precipitation='rain'), HourlyForecast(time='05-27 22:00', cloud_cover=9, transparency=4, seeing=8, lifted_index=-1, wind_speed=2, wind_direction='N', temperature=21.0, humidity=14, precipitation='rain'), Ho

In [5]:
from rich import print as rprint

In [40]:
rprint(results)

{
    'Paris': WeatherReport(
        name='Paris',
        latitude=48.8566,
        longitude=2.3522,
        forecasts=[
            HourlyForecast(
                time='05-26 23:00',
                cloud_cover=1,
                transparency=2,
                seeing=6,
                lifted_index=2,
                wind_speed=2,
                wind_direction='NE',
                temperature=20.0,
                humidity=8,
                precipitation='none'
            ),
            HourlyForecast(
                time='05-27 02:00',
                cloud_cover=1,
                transparency=3,
                seeing=6,
                lifted_index=2,
                wind_speed=2,
                wind_direction='N',
                temperature=18.0,
                humidity=9,
                precipitation='none'
            ),
            HourlyForecast(
                time='05-27 05:00',
                cloud_cover=1,
                transparency=3,
                seeing=6,
                lifted_index=6,
                wind_speed=2,
                wind_direction='NE',
                temperature=16.0,
                humidity=11,
                precipitation='none'
            ),
            HourlyForecast(
                time='05-27 08:00',
                cloud_cover=1,
                transparency=3,
                seeing=3,
                lifted_index=2,
                wind_speed=2,
                wind_direction='NE',
                temperature=20.0,
                humidity=9,
                precipitation='none'
            ),
            HourlyForecast(
                time='05-27 23:00',
                cloud_cover=1,
                transparency=4,
                seeing=5,
                lifted_index=-4,
                wind_speed=2,
                wind_direction='NE',
                temperature=20.0,
                humidity=12,
                precipitation='none'
            ),
            HourlyForecast(
                time='05-28 02:00',
                cloud_cover=1,
                transparency=4,
                seeing=5,
                lifted_index=2,
                wind_speed=2,
                wind_direction='NE',
                temperature=17.0,
                humidity=12,
                precipitation='none'
            ),
            HourlyForecast(
                time='05-28 05:00',
                cloud_cover=6,
                transparency=4,
                seeing=5,
                lifted_index=2,
                wind_speed=2,
                wind_direction='NE',
                temperature=16.0,
                humidity=12,
                precipitation='none'
            ),
            HourlyForecast(
                time='05-28 08:00',
                cloud_cover=7,
                transparency=4,
                seeing=3,
                lifted_index=2,
                wind_speed=2,
                wind_direction='NE',
                temperature=18.0,
                humidity=10,
                precipitation='none'
            )
        ],
        sunset_time=None,
        sunrise_time=None,
        special_description=None,
        raw_response={
            'product': 'astro',
            'init': '2026052606',
            'dataseries': [
                {
                    'timepoint': 3,
                    'cloudcover': 1,
                    'seeing': 3,
                    'transparency': 2,
                    'lifted_index': 2,
                    'rh2m': 4,
                    'wind10m': {'direction': 'NE', 'speed': 2},
                    'temp2m': 26,
                    'prec_type': 'none'
                },
                {
                    'timepoint': 6,
                    'cloudcover': 2,
                    'seeing': 3,
                    'transparency': 2,
                    'lifted_index': 2,
                    'rh2m': 1,
                    'wind10m': {'direction': 'NE', 'speed': 2},
                    'temp2m': 30,
        

In [3]:
report = default_weather_fn(name="Vienna", latitude=48.2, longitude=16.37, timezone_offset=1)

In [15]:
rprint(report.model_dump())

{
    'name': 'Vienna',
    'latitude': 48.2,
    'longitude': 16.37,
    'forecasts': [
        {
            'time': '05-27 22:00',
            'cloud_cover': 1,
            'transparency': 3,
            'seeing': 5,
            'lifted_index': 6,
            'wind_speed': 2,
            'wind_direction': 'N',
            'temperature': 17.0,
            'humidity': 9,
            'precipitation': 'rain'
        },
        {
            'time': '05-28 01:00',
            'cloud_cover': 1,
            'transparency': 3,
            'seeing': 5,
            'lifted_index': 10,
            'wind_speed': 2,
            'wind_direction': 'NW',
            'temperature': 14.0,
            'humidity': 10,
            'precipitation': 'rain'
        },
        {
            'time': '05-28 04:00',
            'cloud_cover': 2,
            'transparency': 2,
            'seeing': 6,
            'lifted_index': 15,
            'wind_speed': 2,
            'wind_direction': 'NW',
            'temperature': 12.0,
            'humidity': 9,
            'precipitation': 'none'
        },
        {
            'time': '05-28 07:00',
            'cloud_cover': 4,
            'transparency': 2,
            'seeing': 2,
            'lifted_index': 10,
            'wind_speed': 2,
            'wind_direction': 'NW',
            'temperature': 15.0,
            'humidity': 6,
            'precipitation': 'none'
        },
        {
            'time': '05-28 22:00',
            'cloud_cover': 2,
            'transparency': 2,
            'seeing': 6,
            'lifted_index': 10,
            'wind_speed': 2,
            'wind_direction': 'NW',
            'temperature': 14.0,
            'humidity': 6,
            'precipitation': 'none'
        },
        {
            'time': '05-29 01:00',
            'cloud_cover': 2,
            'transparency': 2,
            'seeing': 6,
            'lifted_index': 15,
            'wind_speed': 2,
            'wind_direction': 'NW',
            'temperature': 12.0,
            'humidity': 8,
            'precipitation': 'none'
        },
        {
            'time': '05-29 04:00',
            'cloud_cover': 4,
            'transparency': 2,
            'seeing': 6,
            'lifted_index': 15,
            'wind_speed': 2,
            'wind_direction': 'NW',
            'temperature': 10.0,
            'humidity': 9,
            'precipitation': 'none'
        },
        {
            'time': '05-29 07:00',
            'cloud_cover': 3,
            'transparency': 2,
            'seeing': 2,
            'lifted_index': 10,
            'wind_speed': 2,
            'wind_direction': 'NW',
            'temperature': 15.0,
            'humidity': 6,
            'precipitation': 'none'
        }
    ],
    'sunset_time': None,
    'sunrise_time': None,
    'special_description': None
}

# isolated call for searchagent 
### with given capabilities for Search and Weather call

In [ ]:
agent_spot_searcher